# Step 1: prompt conditions C0, C1a, C1b

Scores every prompt (format × render × language × control × option order) with letter log-probs and
compares no system prompt (C0), a simple persona (C1a) and a rich persona (C1b).

**Mac:** run from the repo's `notebooks/` folder. **Kaggle:** add the repo as a Dataset, turn Internet on,
pick GPU T4 x2, `pip install stopwordsiso`, and point `REPO` / `OUT` at `/kaggle/input/...` / `/kaggle/working`.

Scores are saved per condition; re-running loads them instead of scoring again (delete the file to redo it).

In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
import os
import sys
from pathlib import Path

os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")  # Mac: CPU fallback for any op MPS lacks

REPO = Path.cwd().parent  # Kaggle: Path("/kaggle/input/<dataset>/probad")
OUT = REPO / "results"  # Kaggle: Path("/kaggle/working/results")
sys.path.insert(0, str(REPO))

import pandas as pd

pd.set_option("display.width", 160)

In [3]:
MODEL_NAME = "allenai/OLMo-2-0425-1B"  # base model
CHAT = False  # True only for instruct models; keep it fixed across conditions
ITEMS = REPO / "examples" / "sample_items.jsonl"  # later: data/test_v1.jsonl
CONDITIONS = ["C0", "C1a", "C1b"]
BUILD = dict(
    formats=("normal", "negative", "none", "meaning"),
    renders=("romanised", "gloss", "respelled", "bengali"),
    langs=("en", "bn"),
    controls=("none", "no_question"),
)

## Items and prompts

In [4]:
from probad.builder import BuildConfig, PromptBuilder
from probad.schema import TestItem, validate_items
from probad.utils import load_bengali_stopwords, print_div

items = list(TestItem.load_items(ITEMS))
for issue in validate_items(items, load_bengali_stopwords()):
    print(issue)
print_div()

instances = PromptBuilder(BuildConfig(**BUILD)).build_many(items)
print(instances[0].prompt)
print_div()

print(instances[0].cloze_prompt)
print_div()

counts = pd.DataFrame([{"lang": i.lang, "render": i.render, "format": i.format} for i in instances])
print(len(instances), "instances")
print(len(items), "items ->", len(instances), "prompts\n")
print_div()

display(counts.value_counts().unstack(fill_value=0))

[WARNING] bn_0002: bn.literal_copies_proverb: Bengali literal distractor shares 3 content words (75%) with the proverb; paraphrase where possible
Ma: Someone finished the rosogollas I was saving for the guests.
Rahul: Don't look at me, I haven't even opened the fridge today!
Didi: Thakur ghore ke? Ami kola khai na.

Question: What does Didi mean?
A. Rahul's eager denial, before anyone blamed him, shows that he ate the sweets.
B. Someone in the prayer room is saying they do not eat bananas.
Answer:
Ma: Someone finished the rosogollas I was saving for the guests.
Rahul: Don't look at me, I haven't even opened the fridge today!
Didi: Thakur ghore ke? Ami kola khai na.

Question: What does Didi mean?
Answer:
256 instances
2 items -> 256 prompts



format          normal  negative  none  meaning
lang render                                    
en   romanised       8         8     8        8
     gloss           8         8     8        8
     respelled       8         8     8        8
     bengali         8         8     8        8
bn   romanised       8         8     8        8
     gloss           8         8     8        8
     respelled       8         8     8        8
     bengali         8         8     8        8

## Model and scoring

In [ ]:
from probad.constants import SYSTEM_PROMPTS
from probad.scoring import load_model, score

model, tok = load_model(MODEL_NAME)
print(model.device, model.dtype)

/Users/koyeldas/Documents/projects/bn-proverbs-in-context/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

In [ ]:
tag = Path(MODEL_NAME).name
results = pd.concat(
    [score(model, tok, instances, c, SYSTEM_PROMPTS[c], chat=CHAT, model_name=tag, out=OUT / f"{tag}_{c}.jsonl") for c in CONDITIONS]
)
results[["condition", "instance_id", "p_correct", "p_literal", "pred_key", "correct", "margin"]].head()

In [ ]:
# Sanity check: probability mass on the answer letters. Low values = the model isn't answering with a letter.
results.groupby("condition")["letter_mass"].describe()[["mean", "min", "50%"]]

## Results

`item_level` collapses the option orders (averaged: mean option probability over orders; strict: right in
every order). The headline setting is English conversation, romanised proverb, question shown.

In [ ]:
from probad.constants import FORMAT_CHANCE
from probad.metrics import accuracy, compare, item_level

items_avg = item_level(results)
items_strict = item_level(results, strict=True)


def headline(df):
    return df[(df.lang == "en") & (df.render == "romanised") & (df.control == "none")]


acc = accuracy(headline(items_avg), by=["condition", "format"])
strict = accuracy(headline(items_strict), by=["condition", "format"])
acc.merge(strict[["condition", "format", "acc"]], on=["condition", "format"], suffixes=("", "_strict")).round(3)

In [ ]:
import matplotlib.pyplot as plt

COLORS = {"C0": "#2a78d6", "C1a": "#eb6834", "C1b": "#1baf7a"}
formats = list(dict.fromkeys(acc["format"]))
width = 0.8 / len(CONDITIONS)

fig, ax = plt.subplots(figsize=(7, 3.5))
for k, c in enumerate(CONDITIONS):
    sub = acc[acc.condition == c].set_index("format").reindex(formats)
    x = [i + (k - (len(CONDITIONS) - 1) / 2) * width for i in range(len(formats))]
    ax.bar(x, sub["acc"], width * 0.9, color=COLORS[c], label=c)
    ax.errorbar(x, sub["acc"], yerr=[sub["acc"] - sub["ci_low"], sub["ci_high"] - sub["acc"]], fmt="none", ecolor="#52514e", elinewidth=1)
for i, f in enumerate(formats):
    ax.hlines(FORMAT_CHANCE[f], i - 0.45, i + 0.45, color="#8a8984", linestyle="--", linewidth=1)
ax.set_xticks(range(len(formats)), formats)
ax.set_ylim(0, 1)
ax.set_ylabel("accuracy (95% CI)")
ax.set_title("English conversation, romanised proverb (dashed line = chance)", loc="left", fontsize=10)
ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False, loc="upper left", bbox_to_anchor=(1, 1))
plt.tight_layout()
plt.show()

In [ ]:
# Does a persona help? Paired on the same items: difference, bootstrap CI, McNemar exact p-value.
pd.concat([compare(headline(items_avg), c, "C0", by=["format"]) for c in CONDITIONS if c != "C0"]).round(3)

## Breakdowns

In [ ]:
# Render: does the romanised form itself hurt? (normal format, English conversation)
normal = items_avg[(items_avg.lang == "en") & (items_avg.control == "none") & (items_avg.format == "normal")]
accuracy(normal, by=["condition", "render"]).round(3)

In [ ]:
# Figurative vs literal proverbs, and proverbs with English look-alike words (romanised, normal format)
romanised = normal[normal.render == "romanised"]
display(accuracy(romanised, by=["condition", "figurative"]).round(3))
accuracy(romanised, by=["condition", "has_false_friends"]).round(3)

In [ ]:
# Bengali conversation vs English conversation (Bengali-script proverb in both)
accuracy(items_avg[(items_avg.render == "bengali") & (items_avg.control == "none")], by=["condition", "lang", "format"]).round(3)

In [ ]:
# Control: with the question removed, accuracy should fall toward chance if the model uses the question
accuracy(items_avg[(items_avg.lang == "en") & (items_avg.render == "romanised")], by=["condition", "control", "format"]).round(3)